# PLD and percolation (winding method)

In this notebook we will

- compute the Largest Cavity Diameter (LCD), Pore Limiting Diameter (PLD) and Largest Included Sphere (LIS) with the **winding-number method**, the standard percolation method in mofography,
- read the percolation history that the PLD is estimated from,
- visualize the percolating clusters at a given probe size,
- compare against the older **face-merging** method at the end (comparison only).

The winding method works on the pore graph: nodes are pockets, edges are windows carrying a periodic cell offset. A cluster percolates along an axis if it contains a cycle whose accumulated offset (winding number) is non-zero along that axis. This is exact for contorted channels that wrap around the cell obliquely. See [Face merging vs winding](face_merging_vs_winding.qmd) and [Accessibility](accessibility.qmd) for the background.

Prerequisites: [distance grid](distance_grid.qmd) and [region segmentation](region_segmentation.qmd).

## Imports

In [1]:
import warnings, logging
warnings.filterwarnings("ignore")
logging.disable(logging.WARNING)
import numpy as np
from ase.io import read
import mofography as mgr

## Segment the structure

The winding functions take the periodic segmentation `seg` (with its distance grid), so we build it once and reuse it.

The example structure is **NU-1000** (`examples/TestCIFs/NU-1000_S1.cif`), a mesoporous MOF with large hexagonal channels along z, at grid spacing 0.5 Å.

In [2]:
atoms = read("../examples/TestCIFs/NU-1000_S1.cif")   # NU-1000 (examples/TestCIFs)
spacing = 0.5
dgrid = mgr.dgrid_from_atoms_rust(atoms, spacing=spacing)
seg = mgr.do_periodic_segmentation_rust(
    dgrid.astype(np.float32), atoms,
    mask_thickness=0.05, h=0.2, min_distance=1.5, threshold_abs=1.5,
    spacing=spacing, compute_connections=True, return_dgrid=True, disable_tqdm=True,
)
print("pockets (PBC-grouped):", len(seg["maxima_pbc"]))

[RAYON] current_num_threads = 14
[RAYON] current_num_threads = 14


pockets (PBC-grouped): 11


## LCD, PLD and LIS (winding)

`get_pore_dia_mgr_winding` is the one-call entry point. It returns `(lcd, pld, lis, dgrid)`, all diameters in Å. PLD is the largest probe diameter for which a percolating winding cycle exists; LIS is the largest included sphere along the percolating pathways. (Older code calls the third value `dfsd`.)

> There is no function called `compute_pore_diameters_winding`; `get_pore_dia_mgr_winding` is the actual name.

In [3]:
lcd, pld, lis, _ = mgr.get_pore_dia_mgr_winding(results=seg, mat_atoms=atoms, include_dangling=True)
print(f"LCD = {lcd:.3f} Å   PLD = {pld:.3f} Å   LIS = {lis:.3f} Å")

LCD = 29.415 Å   PLD = 28.815 Å   LIS = 29.415 Å


## Percolation history (winding)

`compute_percolation_history_winding` sweeps the probe radius and returns `(percolation_history, pld_locations, final_clusters)`. Each history entry is `(probe_radius, {'x': [...], 'y': [...], 'z': [...]})`, where each axis lists the percolating clusters (lists of pocket labels). `step` is the sweep step in Å (probe radius).

In [4]:
hist, pld_locations, clusters = mgr.compute_percolation_history_winding(seg, step=0.1)
print("entries:", len(hist), " first probe radius:", hist[0][0], " last:", round(hist[-1][0], 2))
print("percolating clusters at the last probe radius (pockets per cluster):")
for axis, cl in hist[-1][1].items():
    print(" ", axis, [len(x) for x in cl])

entries: 145  first probe radius: 0.0  last: 14.4
percolating clusters at the last probe radius (pockets per cluster):
  x []
  y []
  z [8]


The last entry is the largest probe radius for which some axis still percolates; PLD/2 should match it to within the sweep step. For NU-1000 the last radius is about 14.4 Å (PLD about 28.8 Å), and only the z axis still percolates: the large channels run along z, and the narrower connections in the xy plane close at smaller probes.

In [5]:
print(f"PLD/2 = {pld/2:.3f} Å, last percolating radius = {hist[-1][0]:.3f} Å")
assert abs(pld/2 - hist[-1][0]) <= 0.1 + 1e-6

PLD/2 = 14.407 Å, last percolating radius = 14.400 Å


## Visualize the percolating clusters at the PLD probe

The percolation history is ordered by increasing probe radius, so its last entry, `hist[-1]`, is the largest probe that still percolates. That radius is the pore limiting radius (PLD/2), and the cluster stored there is what you see below: the last percolating cluster, the pores and windows that stay connected across the cell at the PLD probe. A slightly larger probe breaks it.

In [6]:
import plotly.graph_objects as go
traces = list(mgr.plot_percolated_cluster_meshes(
    hist, seg["dgrid"], seg["regions"], atoms,
    isovalue=1.0, stylize=True, probe_radius=hist[-1][0],
))
cell = mgr.plot_cell(atoms)
traces += list(cell) if isinstance(cell, (list, tuple)) else [cell]
fig = go.Figure(traces)
fig.update_layout(scene_aspectmode="data", height=600, margin=dict(l=0, r=0, t=0, b=0))
fig
fig.write_image("tutorial_images/pld_percolation_clusters.png", width=1000, height=700)  # static preview for the docs


![NU-1000: percolating cluster at the largest percolating probe](tutorial_images/pld_percolation_clusters.png)

::: {.callout-note collapse="true" title="Interactive version"}
```python
fig.show()   # rotatable Plotly figure (the `fig` built above)
# or a standalone Three.js page with one toggleable cluster layer per probe radius
# (see scripts/dev/gen_gallery_assets.py::build_nu1000_percolation)
```
:::

[Open the interactive version](interactive/nu1000_percolation_history_threejs.html){target="_blank"}

## Comparison only: face merging (older, less accurate)

Before the winding method, percolation was decided on the labelled voxel grid by merging labels across periodic faces (the same idea as PoreBlazer-style voxel sweeps). It cannot see paths that leave the cell, wrap around and re-enter, and it is tied to voxel contiguity, so it is **less accurate on contorted frameworks**. It is kept here only for comparison; use the winding functions above for real work.

On NU-1000 the two differ by only about 0.2 Å in PLD (29.0 vs 28.8 Å), which is the size of a grid step effect at spacing 0.5 Å. The gap grows on frameworks whose channels wrap obliquely.

In [7]:
lcd_f, pld_f, lis_f, _ = mgr.get_pore_dia_mgr(atoms, kwargs=dict(spacing=spacing))   # face merging
rows = [("winding (standard)", lcd, pld, lis), ("face merging (legacy)", lcd_f, pld_f, lis_f)]
for name, a, b, c in rows:
    print(f"{name:<24s} LCD {a:7.3f}  PLD {b:7.3f}  LIS {c:7.3f}  Å")
print(f"PLD difference (face - winding): {pld_f - pld:+.3f} Å")

winding (standard)       LCD  29.415  PLD  28.815  LIS  29.415  Å
face merging (legacy)    LCD  29.615  PLD  29.002  LIS  29.615  Å
PLD difference (face - winding): +0.187 Å


## Summary

- Use `get_pore_dia_mgr_winding` for LCD / PLD / LIS and `compute_percolation_history_winding` for the percolation history.
- Winding numbers detect true periodic percolation, including oblique wrap-around channels.
- Face merging (`get_pore_dia_mgr`, lattice histories) is retained only to compare with voxel-based tools.